### Google Colaboratory setup

In [ ]:
#@title 0. SQAIハンズオン環境のセットアップ

from pathlib import Path
import os
import subprocess
import sys


def running_on_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


ON_COLAB = running_on_colab()

if ON_COLAB:
    REPO_URL = "https://github.com/tiksato/SQAI_handson.git"
    REPO_REF = "main"
    PROJECT_ROOT = Path("/content/SQAI_handson")

    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--quiet",
                "--depth",
                "1",
                "--branch",
                REPO_REF,
                REPO_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

else:
    current = Path.cwd().resolve()

    if current.name == "SQAI_notebooks":
        PROJECT_ROOT = current.parent
    elif (current / "pyproject.toml").exists():
        PROJECT_ROOT = current
    else:
        raise RuntimeError(
            "SQAI_handsonのリポジトリルートを特定できません。"
        )

os.chdir(PROJECT_ROOT)

print("Environment:", "Google Colab" if ON_COLAB else "Local Python")
print("Project root:", PROJECT_ROOT)

### Module import

In [ ]:
import numpy as np
from pyscf import gto, scf, fci, cc

### Hartree-Fock, Coupled-Cluster, and Full-CI calculations

In [ ]:
R_list = [1.0, 1.6]

basis_list = [
    'STO-3G', 
]

for R_HH in R_list:
    atoms=f'''
        H {-5*R_HH/2} 0 0
        H {-3*R_HH/2} 0 0
        H   {-R_HH/2} 0 0
        H    {R_HH/2} 0 0
        H  {3*R_HH/2} 0 0
        H  {5*R_HH/2} 0 0
        '''
    print(f"R_HH = {R_HH:.2f}")
    for basis in basis_list:
        mol = gto.M(
            atom=atoms,
            basis=basis, 
            charge=0,
            spin=0,
            symmetry=False, 
            verbose=0,
            )
        HF = scf.RHF(mol).run()
        E_HF = HF.e_tot

        mycc = cc.CCSD(HF)
        e_corr_ccsd, t1, t2 = mycc.kernel()
        E_CCSD = mycc.e_tot
        
        E_FCI, CIC = fci.FCI(HF).kernel()
        
        print(f" Number of basis: {mol.nao:3d}",
              f" HF: {E_HF:.5f}", 
              f" CCSD: {E_CCSD:.5f}", 
              f" FCI: {E_FCI:.5f}")